In [ ]:
# sps_test.ipynb
# 验证实验：空间平滑的两种"补零 + 全阵列项"方案在经典构造下的行为
#   方案 1：每个子阵协方差补零放回 N 维（等价于 M⊙R 三角窗加权）+ β·全阵列协方差
#   方案 2：先经典平滑成 L×L，再补零嵌回 N 维 + β·全阵列协方差
# 预期结论（经实测修正）：
#   方案 1：β 小时只剩三角窗（子空间由截断导向矢量张成，LS-ESPRIT 拟合崩溃，|Ψ| 远离 1），
#           β 大时退化成无平滑——全 β 范围内不优于无平滑
#   方案 2：补零破坏平移不变性 → Ψ 特征值跑出单位圆 → 角度有偏
#   学习版才有能力让子空间/平移结构自洽——本实验为论文 motivation 提供证据
# 单位约定：与 SubspaceNet 一致——计算用弧度，打印/画图用度

import numpy as np
import matplotlib
matplotlib.rc("font", family='YouYuan')
import matplotlib.pyplot as plt

# 场景参数
N, K, L = 6, 2, 4          # 阵元数、源数、子阵长度
P = N - L + 1              # 子阵个数 = 3
T = 200                    # 快拍数
SNR_dB = 10

D2R = np.pi / 180
R2D = 180 / np.pi

def steering(theta_deg):
    """ULA 导向矢量（与 SubspaceNet 一致：d=1/2, f=1，相位 -jπm sinθ）"""
    m = np.arange(N)
    return np.exp(-1j * np.pi * m * np.sin(theta_deg * D2R))

def make_data(seed):
    """生成两个完全相干源的快拍和样本协方差"""
    rng = np.random.default_rng(seed)
    theta_true = np.array([-10.0, 25.0])
    A = np.zeros((N, K), dtype=complex)
    for k in range(K):
        A[:, k] = steering(theta_true[k])
    sig = (rng.standard_normal(T) + 1j * rng.standard_normal(T)) / np.sqrt(2)
    c = np.array([0.6 + 0.3j, 0.4 - 0.5j])          # 相干系数（固定 → 完全相干）
    sig_power = 10 ** (SNR_dB / 10)
    X = np.sqrt(sig_power) * (A @ (c[:, None] * sig[None, :]))
    noise = (rng.standard_normal((N, T)) + 1j * rng.standard_normal((N, T))) / np.sqrt(2)
    X = X + noise
    R = X @ X.conj().T / T
    return X, R, theta_true

def esprit_from_cov(R, return_psi=False):
    """标准 ESPRIT（numpy 版）：返回角度（度），可选返回 Ψ 特征值以检查单位圆"""
    evals, evecs = np.linalg.eigh(R)
    idx = np.argsort(evals)[::-1][:K]
    Us = evecs[:, idx]
    U1 = Us[:-1, :]
    U2 = Us[1:, :]
    psi = np.linalg.pinv(U1) @ U2
    psi_evals = np.linalg.eigvals(psi)
    doa = -np.arcsin(np.angle(psi_evals) / np.pi) * R2D
    if return_psi:
        return doa, psi_evals
    return doa

def rmsp_deg(pred, true):
    """周期 RMSE（度）：K=2 时只需比较两种排列"""
    best = 1e9
    for perm in [pred, pred[::-1]]:
        err = ((perm - true + 90) % 180) - 90
        err_val = np.sqrt(np.mean(err ** 2))
        best = min(best, err_val)
    return best

print('场景: N=%d, K=%d 相干源, L=%d, P=%d, T=%d, SNR=%ddB' % (N, K, L, P, T, SNR_dB))

In [ ]:
# 单次实验演示：看各方法的估计角度
X, R, theta_true = make_data(0)

# 基线 1：无平滑（相干源下应失效）
doa_no = np.sort(esprit_from_cov(R))

# 基线 2：前向空间平滑（塌缩成 L×L）
R_f = np.zeros((L, L), dtype=complex)
for p in range(P):
    R_f = R_f + R[p:p + L, p:p + L]
R_f = R_f / P
doa_ss = np.sort(esprit_from_cov(R_f))

# 基线 3：前后向平滑 FBSS
J = np.fliplr(np.eye(L))
R_fb = (R_f + J @ R_f.conj() @ J) / 2
doa_fbss = np.sort(esprit_from_cov(R_fb))

# 方案 1（β=1 示例）：三角窗加权 + 全阵列项
M_win = np.zeros((N, N))
for p in range(P):
    M_win[p:p + L, p:p + L] = M_win[p:p + L, p:p + L] + 1.0
R_m1 = M_win * R + 1.0 * R
doa_m1 = np.sort(esprit_from_cov(R_m1))

# 方案 2（β=1 示例）：先平滑后补零 + 全阵列项
R_f_pad = np.zeros((N, N), dtype=complex)
R_f_pad[:L, :L] = R_f
R_m2 = R_f_pad + 1.0 * R
doa_m2, psi_m2 = esprit_from_cov(R_m2, return_psi=True)

print('真实角度(度):', theta_true)
print('无平滑 ESPRIT:', np.round(doa_no, 2))
print('前向平滑 ESPRIT:', np.round(doa_ss, 2))
print('FBSS ESPRIT:', np.round(doa_fbss, 2))
print('方案1(β=1) ESPRIT:', np.round(doa_m1, 2))
print('方案2(β=1) ESPRIT:', np.round(doa_m2, 2))
print('方案2 Ψ特征值模:', np.round(np.abs(psi_m2), 3), ' (理想值应≈1)')

In [ ]:
# 蒙特卡洛扫描：RMSPE 与 Ψ 特征值随 β 变化
betas = np.logspace(-3, 2, 21)
MC = 5
rmsp_no_list = []
rmsp_ss_list = []
rmsp_fbss_list = []
rmsp_m1 = np.zeros((MC, len(betas)))
rmsp_m2 = np.zeros((MC, len(betas)))
psi_mod_m1 = np.zeros((MC, len(betas), K))
psi_mod_m2 = np.zeros((MC, len(betas), K))

for mc in range(MC):
    X, R, theta_true = make_data(mc)
    rmsp_no_list.append(rmsp_deg(np.sort(esprit_from_cov(R)), theta_true))
    R_f = np.zeros((L, L), dtype=complex)
    for p in range(P):
        R_f = R_f + R[p:p + L, p:p + L]
    R_f = R_f / P
    rmsp_ss_list.append(rmsp_deg(np.sort(esprit_from_cov(R_f)), theta_true))
    J = np.fliplr(np.eye(L))
    R_fb = (R_f + J @ R_f.conj() @ J) / 2
    rmsp_fbss_list.append(rmsp_deg(np.sort(esprit_from_cov(R_fb)), theta_true))
    M_win = np.zeros((N, N))
    for p in range(P):
        M_win[p:p + L, p:p + L] = M_win[p:p + L, p:p + L] + 1.0
    R_f_pad = np.zeros((N, N), dtype=complex)
    R_f_pad[:L, :L] = R_f
    for bi in range(len(betas)):
        beta = betas[bi]
        doa1, psi1 = esprit_from_cov(M_win * R + beta * R, return_psi=True)
        rmsp_m1[mc, bi] = rmsp_deg(np.sort(doa1), theta_true)
        psi_mod_m1[mc, bi, :] = np.abs(psi1)
        doa2, psi2 = esprit_from_cov(R_f_pad + beta * R, return_psi=True)
        rmsp_m2[mc, bi] = rmsp_deg(np.sort(doa2), theta_true)
        psi_mod_m2[mc, bi, :] = np.abs(psi2)

print('基线 RMSPE(度, %d 次均值): 无平滑 %.2f, 前向平滑 %.2f, FBSS %.2f' % (
    MC, np.mean(rmsp_no_list), np.mean(rmsp_ss_list), np.mean(rmsp_fbss_list)))

In [ ]:
# 图 1：两种方案的 RMSPE 随 β 变化（横轴对数），横线为三个基线
plt.figure(figsize=(9, 5))
plt.plot(betas, rmsp_m1.mean(axis=0), 'o-', label='方案1: M⊙R + βR')
plt.plot(betas, rmsp_m2.mean(axis=0), 's--', label='方案2: 补零(R_f) + βR')
plt.axhline(np.mean(rmsp_no_list), color='gray', ls=':', label='无平滑')
plt.axhline(np.mean(rmsp_ss_list), color='green', ls=':', label='前向平滑(L=4)')
plt.axhline(np.mean(rmsp_fbss_list), color='red', ls=':', label='FBSS(L=4)')
plt.xscale('log')
plt.xlabel('β (全阵列项权重)')
plt.ylabel('RMSPE (度)')
plt.legend()
plt.title('方案1/方案2 的 ESPRIT 精度 vs 全阵列项权重（%d 次平均）' % MC)
plt.grid(alpha=0.3)
plt.show()

In [ ]:
# 图 2：两个方案的 Ψ 特征值模随 β 变化（理想值应落在单位圆 |ψ|=1 上）
# 实测结论：两个方案的 |Ψ| 都明显偏离 1 → 平移不变性都被破坏
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for k in range(K):
    axes[0].plot(betas, psi_mod_m1[:, :, k].mean(axis=0), 'o-', label='Ψ特征值' + str(k + 1))
axes[0].axhline(1.0, color='black', ls='--', label='单位圆 |ψ|=1')
axes[0].set_xscale('log')
axes[0].set_xlabel('β (全阵列项权重)')
axes[0].set_ylabel('|Ψ特征值|')
axes[0].set_title('方案1: M⊙R + βR（截断子空间使 LS 拟合崩溃）')
axes[0].legend()
axes[0].grid(alpha=0.3)
for k in range(K):
    axes[1].plot(betas, psi_mod_m2[:, :, k].mean(axis=0), 'o-', label='Ψ特征值' + str(k + 1))
axes[1].axhline(1.0, color='black', ls='--', label='单位圆 |ψ|=1')
axes[1].set_xscale('log')
axes[1].set_xlabel('β (全阵列项权重)')
axes[1].set_ylabel('|Ψ特征值|')
axes[1].set_title('方案2: 补零(R_f) + βR（零元切断平移关系）')
axes[1].legend()
axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 判读要点（实测修正版）

1. **基线对照**：完全相干源下，无平滑 ESPRIT 失效（信号部分秩 1，主方向是满孔径的相干混合 Ac、平移自洽，所以"一好一坏"）；前向平滑与 FBSS（L=4）精确恢复两个角度（本例误差 ~0.07°）。
2. **方案 1（M⊙R + βR）**：秩恢复成立（信号部分秩=3，可验证），但子空间由"截断导向矢量"张成——不满足 ESPRIT 的平移不变性，LS 拟合崩溃：Ψ 特征值模塌缩到 0.2~0.6（应≈1），估计角完全失效；全 β 范围内不优于无平滑（β 大时退化为无平滑）。
   - 注：此前讨论曾预测"U 形曲线、中间优于无平滑"，实测证伪。
3. **方案 2（补零 R_f + βR）**：补零把平移关系切断，|Ψ| 偏离 1，估计有偏。
4. **核心结论（motivation）**：经典平滑只能在 L 维局部坐标里同时保住"秩恢复 + 平移不变性"；任何把 L 维结果"升回" N 维的经典构造（放回相加/taper、补零）都会破坏平移不变性——断法不同（截断子空间崩坏拟合 / 零元切断平移）。因此"无损升维"在经典框架内做不到，学习版（学出相位自洽的 N 维填充）是唯一出路。